## Make sure to run these in Terminal/Conda env
pip install tensorflow tf-keras transformers==4.39.3 jiwer huggingsound crepe pyctcdecode https://github.com/kpu/kenlm/archive/master.zip

## Import required packages

In [1]:
import csv
import os
import pandas as pd
import numpy as np
import wave
import string
import contextlib
import librosa
import torch
import warnings
import contextlib
import json
import jiwer
import kenlm
import torch
from huggingsound import SpeechRecognitionModel, KenshoLMDecoder
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

2025-12-11 01:24:26.157084: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 01:24:26.708412: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-11 01:24:28.013417: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Set directory paths

In [20]:
root_dir = os.getcwd()
rec_dir = './VM-A10/'

In [21]:
def wpm(audio_paths):
    model = SpeechRecognitionModel("jonatasgrosman/wav2vec2-large-xlsr-53-english")
    lm_path = "vm.arpa"
    unigrams_path = "vm_1.arpa"

    print(lm_path)
    decoder = KenshoLMDecoder(model.token_set, lm_path=lm_path, unigrams_path=unigrams_path)
    transcriptions = model.transcribe(audio_paths, decoder=decoder)
    transcriptions2 = model.transcribe(audio_paths)

    print(transcriptions[0])
    trans = transcriptions[0]["transcription"]

    # Count the number of words in the list
    res = sum([i.strip(string.punctuation).isalpha() for i in trans.split()])

    # Get the duration
    start_t = transcriptions2[0]["start_timestamps"][0]
    end_t = transcriptions2[0]["end_timestamps"][-1]
    duration = end_t - start_t
    # Words per minute
    wpm3 = res*60/(duration/1000)
    dur2 = duration/1000
    print("Duration: ")
    print(dur2)
    print("Words per minute: ")
    print(wpm3)

    return [dur2, wpm3, trans]


prompts2 = 'prompts2.csv'

#Open csv containing speech prompts
with open(prompts2, "r", encoding='utf-8-sig') as csv_file2:
    csv_reader2 = csv.reader(csv_file2)
    rows2 = list(csv_reader2)

    # open the file in the write mode
    f = open(rec_dir+'/rec_summary_f0.csv', 'w')
    # create the csv writer
    writer = csv.writer(f)
    header = ["File", "Reference", "Transcription", "Min F0", "Median F0", "Max F0", "Duration", "WPM","WER"]
    # write a row to the csv file
    writer.writerow(header)


for k in range(1,13):

    os.system("crepe "+rec_dir+"rec_"+str(k)+".wav --viterbi")
    crepe_vec = pd.read_csv(rec_dir+'rec_'+str(k)+'.f0.csv')
    crepe_time = pd.DataFrame(crepe_vec.iloc[:,0].to_numpy())
    crepe_freq = pd.DataFrame(crepe_vec.iloc[:,1].to_numpy())
    crepe_conf = pd.DataFrame(crepe_vec.iloc[:,2].to_numpy())
    threshold = 0.75
    conf_thresh = [1 if i>threshold else 0 for i in crepe_conf.to_numpy()]
    freq_thresh = np.transpose(np.asarray(conf_thresh))*np.asarray(crepe_freq).reshape(-1)
    nz_freq = freq_thresh[np.nonzero(freq_thresh)]

    min_f0 = np.min(nz_freq)
    med_f0 = np.median(nz_freq)
    max_f0 = np.max(nz_freq)
    print("The minimum is: ")
    print(min_f0)
    print("The median is: ")
    print(med_f0)
    print("The maximum is: ")
    print(max_f0)

    [dur, w, tr] = wpm([rec_dir+'rec_'+str(k)+'.wav'])
    reftxt = ' '.join(rows2[k-1])
    wer = jiwer.wer(reftxt.lower(),tr)
    print("The word error rate is: ")
    print(wer)
    row = [rec_dir+'rec_'+str(k)+'.wav', reftxt.lower(), tr, str(min_f0), str(med_f0), str(max_f0), dur, w, wer]
    # write a row to the csv file
    writer.writerow(row)

    # close the file
f.close()

CREPE: Processing ./VM-A10/rec_1.wav ... (1/1)
/home/xron/.local/lib/python3.10/site-packages/crepe/core.py:314: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sr, audio = wavfile.read(file)
2025-12-11 02:30:04.442940: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 02:30:04.484479: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-11 02:30:05.073484: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2025-12-11 02:30:05.617579: E external/local_xla/xla/

8/8 ━━━━━━━━━━━━━━━━━━━━ 6s 748ms/step
CREPE: Saved the estimated frequencies and confidence values at ./VM-A10/rec_1.f0.csv
The minimum is: 
191.029
The median is: 
217.85399999999998
The maximum is: 
323.524
12/11/2025 02:30:13 - INFO - huggingsound.speech_recognition.model - Loading model...


Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
Y

vm.arpa
12/11/2025 02:30:15 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
12/11/2025 02:30:15 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
12/11/2025 02:30:15 - WARNING - pyctcdecode.language_model - Only 145 unigrams passed as vocabulary. Is this small or artificial data?
12/11/2025 02:30:15 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


Loading the LM will be faster if you build a binary file.
Reading /mnt/h/My Drive/PhD expts/Voice Match/vm.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.89it/s]


...

59/59 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step
CREPE: Saved the estimated frequencies and confidence values at ./VM-A10/rec_12.f0.csv
The minimum is: 
166.83
The median is: 
196.24450000000002
The maximum is: 
307.533
12/11/2025 02:35:07 - INFO - huggingsound.speech_recognition.model - Loading model...


Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-english and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original1']
Y

vm.arpa
12/11/2025 02:35:09 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
12/11/2025 02:35:09 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
12/11/2025 02:35:09 - WARNING - pyctcdecode.language_model - Only 145 unigrams passed as vocabulary. Is this small or artificial data?
12/11/2025 02:35:09 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


Loading the LM will be faster if you build a binary file.
Reading /mnt/h/My Drive/PhD expts/Voice Match/vm.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.62s/it]

{'transcription': 'the only person for whom the house was in any special way was arthur dent and that was only because it happened to be the one he lived in he lived in it for about three years ever since he had moved out of london because it made him nervous and irritable he was about thirty as well dark haired and never quite at ease with himself', 'start_timestamps': [100, 120, 140, 180, 320, 360, 400, 440, 480, 560, 600, 640, 680, 720, 760, 900, 980, 1000, 1020, 1060, 1120, 1160, 1200, 1240, 1300, 1380, 1400, 1420, 1460, 1500, 1540, 1580, 1620, 1660, 1760, 1820, 1860, 1900, 1980, 2100, 2140, 2180, 2400, 2440, 2480, 2540, 2600, 2640, 2680, 2720, 2760, 2800, 2840, 2920, 2960, 3000, 3040, 3120, 3200, 3240, 3280, 3320, 3460, 3500, 3540, 3580, 3620, 3660, 3700, 3760, 3820, 3880, 3940, 4020, 4780, 4800, 4820, 4840, 4880, 4900, 4920, 4940, 5020, 5060, 5080, 5100, 5160, 5260, 5280, 5300, 5320, 5400, 5440, 5480, 5520, 5560, 5600, 5640, 5680, 5780, 5840, 5860, 5900, 5940, 5960, 5980, 6000, 6

In [2]:
import subprocess, os
root_dir = os.getcwd()
rec_dir = './tts_data/'

In [3]:
def wpm(audio_paths):
    model = SpeechRecognitionModel("jonatasgrosman/wav2vec2-large-xlsr-53-english")
    lm_path = "prompts3.arpa"
    unigrams_path = "prompts3.arpa"

    print(lm_path)
    decoder = KenshoLMDecoder(model.token_set, lm_path=lm_path, unigrams_path=unigrams_path)
    transcriptions = model.transcribe(audio_paths, decoder=decoder)
    transcriptions2 = model.transcribe(audio_paths)
    
    print(transcriptions[0])
    trans = transcriptions[0]["transcription"]

    # Count the number of words in the list
    res = sum([i.strip(string.punctuation).isalpha() for i in trans.split()])

    # Get the duration
    start_t = transcriptions2[0]["start_timestamps"][0]
    end_t = transcriptions2[0]["end_timestamps"][-1]
    duration = end_t - start_t
    # Words per minute
    wpm3 = res*60/(duration/1000)
    dur2 = duration/1000
    print("Duration: ")
    print(dur2)
    print("Words per minute: ")
    print(wpm3)

    return [dur2, wpm3, trans]

def process_participant_data(participants_folder):
    """
    Processes participant data to estimate F0 pitch and speech rate,
    parsing specific file ranges based on the subfolder.
    """
    data = {
        'Participant': [],
        'TTS Voice': [],
        'Mean_F0': [],
        'Std_F0': [],
        'Mean_SpeechRate': [],
        'Std_SpeechRate': []
    }

    for participant_folder in os.listdir(participants_folder):
        participant_path = os.path.join(participants_folder, participant_folder)
        print(participant_path)
        if os.path.isdir(participant_path):
            for subfolder in ['en-US-Standard-B', 'en-US-Standard-C', 'en-US-Neural2-I', 'en-US-Neural2-G']:
                subfolder_path = os.path.join(participant_path, subfolder)
                if os.path.isdir(subfolder_path):
                    f0_values = []
                    speech_rates = []
                    
                    start_index = 0
                    end_index = 0

                    if subfolder == 'en-US-Standard-B':
                        start_index = 11  #Files 11-24
                        end_index = 24
                    elif subfolder == 'en-US-Standard-C':
                        start_index = 25  #Files 25-38
                        end_index = 38
                    elif subfolder == 'en-US-Neural2-G':
                        start_index = 39  #Files 39-52
                        end_index = 52
                    elif subfolder == 'en-US-Neural2-I':
                        start_index = 53  #Files 53-66
                        end_index = 66
                  
                    #filenames = os.listdir(subfolder_path)
                    
                    for i in range(start_index, end_index):
                        filename = str(i) + "_sr_1_f0_0.wav"
                        filepath = os.path.join(subfolder_path, filename)
                        if os.path.isfile(filepath) and filename.endswith(".wav"):
                            # Placeholder - replace with actual F0/speech rate extraction
                            os.system("crepe "+filepath+" --viterbi")
                            crepe_vec = pd.read_csv(filepath[:-4]+'.f0.csv')
                            crepe_time = pd.DataFrame(crepe_vec.iloc[:,0].to_numpy())
                            crepe_freq = pd.DataFrame(crepe_vec.iloc[:,1].to_numpy())
                            crepe_conf = pd.DataFrame(crepe_vec.iloc[:,2].to_numpy())
                            threshold = 0.75
                            conf_thresh = [1 if i>threshold else 0 for i in crepe_conf.to_numpy()]
                            freq_thresh = np.transpose(np.asarray(conf_thresh))*np.asarray(crepe_freq).reshape(-1)
                            nz_freq = freq_thresh[np.nonzero(freq_thresh)]
                            min_f0 = np.min(nz_freq)
                            med_f0 = np.median(nz_freq)
                            max_f0 = np.max(nz_freq)
                            print("The minimum is: ")
                            print(min_f0)
                            print("The median is: ")
                            print(med_f0)
                            print("The maximum is: ")
                            print(max_f0)
                            [dur, w, tr] = wpm([filepath])
                            f0 = med_f0
                            speech_rate = w
                            f0_values.append(f0)
                            speech_rates.append(speech_rate)

                    if f0_values:
                        data['Participant'].append(participant_folder)
                        data['TTS Voice'].append(subfolder)
                        data['Mean_F0'].append(sum(f0_values) / len(f0_values))
                        data['Std_F0'].append(pd.Series(f0_values).std())
                        data['Mean_SpeechRate'].append(sum(speech_rates) / len(speech_rates)) #Using sum and len
                        data['Std_SpeechRate'].append(pd.Series(speech_rates).std()) #Using pandas for std calculation

    return pd.DataFrame(data)

# Example usage
participants_folder = "tts_data"  # Replace with the actual path
result_df = process_participant_data(participants_folder)
result_df
result_df.to_csv('analysis_partial.csv', index=False) 

tts_data/VM-003


CREPE: Processing tts_data/VM-003/en-US-Standard-B/11_sr_1_f0_0.wav ... (1/1)
2025-04-03 23:37:06.797062: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-03 23:37:06.797853: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-03 23:37:06.802344: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-03 23:37:06.811929: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743719826.826025   12942 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory f

13/13 ━━━━━━━━━━━━━━━━━━━━ 10s 729ms/step
CREPE: Saved the estimated frequencies and confidence values at tts_data/VM-003/en-US-Standard-B/11_sr_1_f0_0.f0.csv
The minimum is: 
50.561
The median is: 
138.0115
The maximum is: 
217.302
04/03/2025 23:37:22 - INFO - huggingsound.speech_recognition.model - Loading model...


/home/xron/anaconda3/envs/python39/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


prompts3.arpa
04/03/2025 23:37:26 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
04/03/2025 23:37:26 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
04/03/2025 23:37:26 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


...

11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 889ms/step
CREPE: Saved the estimated frequencies and confidence values at tts_data/VM-006/en-US-Neural2-G/51_sr_1_f0_0.f0.csv
The minimum is: 
32.904
The median is: 
99.075
The maximum is: 
154.436
04/04/2025 00:33:42 - INFO - huggingsound.speech_recognition.model - Loading model...


/home/xron/anaconda3/envs/python39/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


prompts3.arpa
04/04/2025 00:33:43 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
04/04/2025 00:33:43 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
04/04/2025 00:33:43 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


Loading the LM will be faster if you build a binary file.
Reading /mnt/h/My Drive/PhD expts/Voice Match/prompts3.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.30it/s]

{'transcription': 'the storewalls were lined with colored frocks', 'start_timestamps': [60, 80, 100, 140, 220, 300, 380, 460, 540, 620, 700, 780, 860, 940, 1020, 1080, 1100, 1120, 1140, 1220, 1300, 1360, 1420, 1480, 1540, 1640, 1680, 1720, 1760, 1800, 1840, 1900, 1940, 1980, 2020, 2060, 2100, 2140, 2280, 2340, 2420, 2500, 2580, 2660, 2740], 'end_timestamps': None, 'probabilities': [0.9999593496322632, 0.9999947547912598, 1.371747430312098e-06, 2.2884075860929443e-06, 0.9999916553497314, 0.9999912977218628, 3.1230655395120266e-07, 3.239043735447922e-06, 0.003803653409704566, 0.9999525547027588, 1.5994428892440737e-08, 0.99985671043396, 8.19409397081472e-05, 7.432964110876128e-08, 0.9998263716697693, 0.9997560381889343, 1.4161274521029554e-05, 1.6233373401064455e-07, 0.9992642998695374, 0.9996525049209595, 0.9999711513519287, 7.643068400398079e-09, 4.713502921305235e-09, 3.489786948307483e-08, 1.2160369067260035e-07, 0.9611895680427551, 0.9999451637268066, 1.5385812730528414e-05, 0.00023

In [20]:
result_df.to_csv('VM-002_result_df.csv', index=False) 

## This is a new version for the first 10 Harvard sentences only

In [2]:
def wpm(audio_paths):
    model = SpeechRecognitionModel("jonatasgrosman/wav2vec2-large-xlsr-53-english")
    lm_path = "prompts3.arpa"
    unigrams_path = "prompts3.arpa"

    print(lm_path)
    decoder = KenshoLMDecoder(model.token_set, lm_path=lm_path, unigrams_path=unigrams_path)
    transcriptions = model.transcribe(audio_paths, decoder=decoder)
    transcriptions2 = model.transcribe(audio_paths)
    
    print(transcriptions[0])
    trans = transcriptions[0]["transcription"]

    # Count the number of words in the list
    res = sum([i.strip(string.punctuation).isalpha() for i in trans.split()])

    # Get the duration
    start_t = transcriptions2[0]["start_timestamps"][0]
    end_t = transcriptions2[0]["end_timestamps"][-1]
    duration = end_t - start_t
    # Words per minute
    wpm3 = res*60/(duration/1000)
    dur2 = duration/1000
    print("Duration: ")
    print(dur2)
    print("Words per minute: ")
    print(wpm3)

    return [dur2, wpm3, trans]

def process_participant_data(participants_folder):
    """
    Processes participant data to estimate F0 pitch and speech rate,
    parsing specific file ranges based on the subfolder.
    """
    data = {
        'Participant': [],
        'TTS Voice': [],
        'Mean_F0': [],
        'Std_F0': [],
        'Mean_SpeechRate': [],
        'Std_SpeechRate': []
    }

    for participant_folder in os.listdir(participants_folder):
        participant_path = os.path.join(participants_folder, participant_folder)
        print(participant_path)
        if os.path.isdir(participant_path):
            for subfolder in ['en-US-Standard-B', 'en-US-Standard-C', 'en-US-Neural2-I', 'en-US-Neural2-G']:
                subfolder_path = os.path.join(participant_path, subfolder)
                if os.path.isdir(subfolder_path):
                    f0_values = []
                    speech_rates = []
                    
                    start_index = 1
                    end_index = 10

                    #filenames = os.listdir(subfolder_path)
                    
                    for i in range(start_index, end_index):
                        filename = str(i) + "_sr_1_f0_0.wav"
                        filepath = os.path.join(subfolder_path, filename)
                        if os.path.isfile(filepath) and filename.endswith(".wav"):
                            # Placeholder - replace with actual F0/speech rate extraction
                            os.system("crepe "+filepath+" --viterbi")
                            crepe_vec = pd.read_csv(filepath[:-4]+'.f0.csv')
                            crepe_time = pd.DataFrame(crepe_vec.iloc[:,0].to_numpy())
                            crepe_freq = pd.DataFrame(crepe_vec.iloc[:,1].to_numpy())
                            crepe_conf = pd.DataFrame(crepe_vec.iloc[:,2].to_numpy())
                            threshold = 0.75
                            conf_thresh = [1 if i>threshold else 0 for i in crepe_conf.to_numpy()]
                            freq_thresh = np.transpose(np.asarray(conf_thresh))*np.asarray(crepe_freq).reshape(-1)
                            nz_freq = freq_thresh[np.nonzero(freq_thresh)]
                            min_f0 = np.min(nz_freq)
                            med_f0 = np.median(nz_freq)
                            max_f0 = np.max(nz_freq)
                            print("The minimum is: ")
                            print(min_f0)
                            print("The median is: ")
                            print(med_f0)
                            print("The maximum is: ")
                            print(max_f0)
                            [dur, w, tr] = wpm([filepath])
                            f0 = med_f0
                            speech_rate = w
                            f0_values.append(f0)
                            speech_rates.append(speech_rate)

                    if f0_values:
                        data['Participant'].append(participant_folder)
                        data['TTS Voice'].append(subfolder)
                        data['Mean_F0'].append(sum(f0_values) / len(f0_values))
                        data['Std_F0'].append(pd.Series(f0_values).std())
                        data['Mean_SpeechRate'].append(sum(speech_rates) / len(speech_rates)) #Using sum and len
                        data['Std_SpeechRate'].append(pd.Series(speech_rates).std()) #Using pandas for std calculation

    return pd.DataFrame(data)

# Example usage
participants_folder = "tts_data2"  # Replace with the actual path
result_df = process_participant_data(participants_folder)
result_df
result_df.to_csv('analysis_partial2.csv', index=False) 

tts_data2/VM-003


CREPE: Processing tts_data2/VM-003/en-US-Standard-B/1_sr_1_f0_0.wav ... (1/1)
2025-04-04 03:10:27.562451: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 03:10:27.563276: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 03:10:27.566850: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 03:10:27.577085: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743732627.595234   45477 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory f

12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 764ms/step
CREPE: Saved the estimated frequencies and confidence values at tts_data2/VM-003/en-US-Standard-B/1_sr_1_f0_0.f0.csv
The minimum is: 
51.21
The median is: 
142.60399999999998
The maximum is: 
210.724
04/04/2025 03:10:42 - INFO - huggingsound.speech_recognition.model - Loading model...


/home/xron/anaconda3/envs/python39/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


prompts3.arpa
04/04/2025 03:10:46 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
04/04/2025 03:10:46 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
04/04/2025 03:10:46 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


...

12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 1s/step
CREPE: Saved the estimated frequencies and confidence values at tts_data2/VM-003/en-US-Neural2-G/9_sr_1_f0_0.f0.csv
The minimum is: 
82.504
The median is: 
146.612
The maximum is: 
203.784
04/04/2025 03:21:55 - INFO - huggingsound.speech_recognition.model - Loading model...


/home/xron/anaconda3/envs/python39/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


prompts3.arpa
04/04/2025 03:21:56 - WARNING - pyctcdecode.alphabet - Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
04/04/2025 03:21:56 - WARNING - pyctcdecode.alphabet - Unigrams and labels don't seem to agree.
04/04/2025 03:21:56 - WARNING - pyctcdecode.language_model - Only 0.0% of unigrams in vocabulary found in kenlm model-- this might mean that your vocabulary and language model are incompatible. Is this intentional?


Loading the LM will be faster if you build a binary file.
Reading /mnt/h/My Drive/PhD expts/Voice Match/prompts3.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.33it/s]

{'transcription': 'soap can wash most dirt away', 'start_timestamps': [140, 240, 340, 440, 540, 680, 740, 800, 860, 980, 1080, 1180, 1280, 1440, 1660, 1740, 1820, 1900, 2020, 2200, 2280, 2360, 2440, 2540, 2640, 2740, 2840, 2940], 'end_timestamps': None, 'probabilities': [0.9999426603317261, 2.577241104972927e-07, 3.928765934801959e-08, 2.2773019736632705e-07, 6.610299010390008e-07, 0.9999916553497314, 1.3285823996511681e-08, 5.535185110261409e-08, 3.911394230726728e-07, 0.9999760389328003, 4.931629771220969e-09, 5.234028677136848e-08, 1.1017660561662979e-08, 5.7992122037831e-07, 0.9999887943267822, 1.1464648963510626e-07, 2.5325419983346364e-08, 5.284910571390355e-07, 4.702907290266012e-07, 0.9999873638153076, 6.220067128026585e-09, 3.490388564841851e-07, 7.594543660616182e-09, 4.2933665099553764e-05, 0.9997681975364685, 0.9923248291015625, 8.17465295455122e-09, 9.28562045032777e-08]}
Duration: 
2.92
Words per minute: 
123.28767123287672
tts_data2/desktop.ini
